# MovieLens-100K Novelty Experiments (PopDebias + ColdBridge)

This notebook runs the novelty pipeline described in `kaggle/novelty` on MovieLens-100K using GPU-friendly BGE-M3 embeddings.

It will generate:
- `results/movielens_100k/full_results.csv`
- `results/movielens_100k/best_model.txt`
- `results/figures/*.png`

In [ ]:
import importlib.util
import subprocess
import sys

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is None:
        pip_install(pip_name or import_name)

# Only install what is missing to reduce Kaggle startup time.
ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("sklearn", "scikit-learn")
ensure_package("tensorflow", "tensorflow==2.19.0")
ensure_package("torch")
ensure_package("transformers")
ensure_package("sentencepiece")

print("Dependency check complete.")

In [ ]:
import math
import random
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.decomposition import PCA
from tensorflow import keras
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

WORK_DIR = Path("/kaggle/working/llmseqrec_ml100k_single_notebook")
DATA_DIR = WORK_DIR / "data"
EMB_DIR = WORK_DIR / "embeddings" / "movielens_100k"
RESULTS_DIR = WORK_DIR / "results" / "movielens_100k"
FIG_DIR = WORK_DIR / "results" / "figures"

for p in [DATA_DIR, EMB_DIR, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
SESSION_GAP_MINUTES = 30
MIN_SESSION_LEN = 2
VAL_FRAC = 0.1
TEST_FRAC = 0.2
SEQ_LEN = 20

NUM_EPOCHS = 6
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EMB_DIM = 64

TOP_K = 20
CANDIDATE_K = 300
ALPHAS = [0.1, 0.3, 0.5, 0.7]
TAUS = [2, 3, 5, 10, 15]
DECAYS = [0.5, 0.7, 0.8, 0.9, 1.0]
LONG_TAIL_THRESHOLD = 500

print("Using work dir:", WORK_DIR)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", gpus)

In [ ]:
def download_ml100k(data_dir: Path, dataset_url: str) -> tuple[Path, Path]:
    udata_path = data_dir / "ml-100k" / "u.data"
    uitem_path = data_dir / "ml-100k" / "u.item"
    if udata_path.exists() and uitem_path.exists():
        return udata_path, uitem_path

    data_dir.mkdir(parents=True, exist_ok=True)
    archive_path = data_dir / "ml-100k.zip"
    if not archive_path.exists():
        urllib.request.urlretrieve(dataset_url, archive_path)

    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(data_dir)

    if not (udata_path.exists() and uitem_path.exists()):
        raise FileNotFoundError("MovieLens-100K files not found after extraction.")
    return udata_path, uitem_path


def load_interactions(udata_path: Path) -> pd.DataFrame:
    return pd.read_csv(
        udata_path,
        sep="\t",
        names=["UserId", "ItemId", "Rating", "Timestamp"],
        engine="python",
    )


def sessionize(interactions: pd.DataFrame, gap_minutes: int, min_session_len: int) -> pd.DataFrame:
    df = interactions.copy()
    df["Timestamp"] = df["Timestamp"].astype(int)
    df["ItemId"] = df["ItemId"].astype(int)
    df = df.sort_values(["UserId", "Timestamp", "ItemId"]).reset_index(drop=True)

    gap_seconds = gap_minutes * 60
    user_changed = df["UserId"].ne(df["UserId"].shift(1))
    ts_gap = df["Timestamp"].diff().fillna(0)
    new_session = user_changed | (ts_gap > gap_seconds)
    df["SessionId"] = new_session.cumsum().astype(int)

    valid = df.groupby("SessionId").size()
    valid_ids = valid[valid >= min_session_len].index
    df = df[df["SessionId"].isin(valid_ids)].copy()

    remap = {old: i + 1 for i, old in enumerate(sorted(df["SessionId"].unique()))}
    df["SessionId"] = df["SessionId"].map(remap).astype(int)

    df["Time"] = pd.to_datetime(df["Timestamp"], unit="s", utc=True).dt.tz_localize(None)
    df["Time"] = df["Time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    df["Reward"] = df["Rating"].astype(float)
    return df[["SessionId", "ItemId", "Time", "Reward"]].sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)


def temporal_split(sessions_df: pd.DataFrame, val_frac: float, test_frac: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = sessions_df.copy()
    df["Time"] = pd.to_datetime(df["Time"])

    session_end = df.groupby("SessionId")["Time"].max().sort_values()
    session_ids = session_end.index.to_numpy()
    n = len(session_ids)

    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))
    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("Split produced empty train partition.")

    train_ids = set(session_ids[:n_train])
    val_ids = set(session_ids[n_train:n_train + n_val])
    test_ids = set(session_ids[n_train + n_val:])

    train_df = df[df["SessionId"].isin(train_ids)].copy()
    val_df = df[df["SessionId"].isin(val_ids)].copy()
    test_df = df[df["SessionId"].isin(test_ids)].copy()

    return (
        train_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
        val_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
        test_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
    )


def build_cases(split_df: pd.DataFrame, n_withheld: int = 1) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray], dict[int, int]]:
    prompts: dict[int, np.ndarray] = {}
    gts: dict[int, np.ndarray] = {}
    lengths: dict[int, int] = {}

    for sid, cur in split_df.groupby("SessionId"):
        items = cur["ItemId"].to_numpy(dtype=int)
        if len(items) <= n_withheld:
            continue
        prompt = items[:-n_withheld]
        gt = items[-n_withheld:]
        if len(prompt) == 0:
            continue
        sid = int(sid)
        prompts[sid] = prompt
        gts[sid] = gt
        lengths[sid] = len(prompt)
    return prompts, gts, lengths


def load_item_metadata(uitem_path: Path) -> pd.DataFrame:
    genre_cols = [
        "unknown", "action", "adventure", "animation", "childrens", "comedy",
        "crime", "documentary", "drama", "fantasy", "film_noir", "horror",
        "musical", "mystery", "romance", "sci_fi", "thriller", "war", "western",
    ]
    cols = ["ItemId", "title", "release_date", "video_release_date", "imdb_url", *genre_cols]
    df = pd.read_csv(
        uitem_path,
        sep="|",
        names=cols,
        encoding="latin-1",
        usecols=list(range(len(cols))),
        engine="python",
    )

    def build_text(row: pd.Series) -> str:
        genres = [g.replace("_", " ") for g in genre_cols if int(row[g]) == 1]
        genre_text = ", ".join(genres) if genres else "unknown"
        return f"{row['title']}. Genres: {genre_text}."

    df["item_text"] = df.apply(build_text, axis=1)
    return df[["ItemId", "title", "item_text"]]


def normalize_rows(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def encode_bge_cpu(item_ids: np.ndarray, item_text_df: pd.DataFrame, output_npy: Path, batch_size: int = 32, max_length: int = 256) -> np.ndarray:
    output_npy.parent.mkdir(parents=True, exist_ok=True)

    text_lookup = item_text_df.set_index("ItemId")["item_text"].to_dict()
    texts = [text_lookup.get(int(i), f"Movie item {int(i)}.") for i in item_ids]

    if output_npy.exists():
        cached = np.load(output_npy)
        if cached.shape[0] == len(item_ids):
            return cached

    import torch
    import torch.nn.functional as F

    device = "cpu"
    tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

    try:
        model = AutoModel.from_pretrained("BAAI/bge-m3", dtype=torch.float32)
    except TypeError:
        model = AutoModel.from_pretrained("BAAI/bge-m3", torch_dtype=torch.float32)
    model.to(device)
    model.eval()

    all_chunks = []
    for start in range(0, len(texts), batch_size):
        cur = texts[start:start + batch_size]
        encoded = tokenizer(
            cur,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            out = model(**encoded)
            token_emb = out.last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).expand(token_emb.size()).float()
            pooled = (token_emb * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
            pooled = F.normalize(pooled, p=2, dim=1)

        all_chunks.append(pooled.cpu().numpy().astype(np.float32))

    embeddings = np.vstack(all_chunks).astype(np.float32)
    np.save(output_npy, embeddings)
    return embeddings


def build_id_maps(all_item_ids: np.ndarray) -> tuple[dict[int, int], dict[int, int]]:
    item_to_reduced = {int(item): idx + 1 for idx, item in enumerate(all_item_ids.tolist())}
    reduced_to_item = {idx + 1: int(item) for idx, item in enumerate(all_item_ids.tolist())}
    return item_to_reduced, reduced_to_item


def to_reduced_sequence(seq: np.ndarray, item_to_reduced: dict[int, int]) -> list[int]:
    return [item_to_reduced[int(i)] for i in seq if int(i) in item_to_reduced]


def build_train_examples(train_df: pd.DataFrame, item_to_reduced: dict[int, int], seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    x_rows: list[np.ndarray] = []
    y_rows: list[int] = []

    for _, cur in train_df.groupby("SessionId"):
        items = to_reduced_sequence(cur["ItemId"].to_numpy(dtype=int), item_to_reduced)
        if len(items) < 2:
            continue
        for t in range(1, len(items)):
            prompt = items[max(0, t - seq_len):t]
            target = items[t]
            x = np.zeros(seq_len, dtype=np.int32)
            x[-len(prompt):] = np.array(prompt, dtype=np.int32)
            x_rows.append(x)
            y_rows.append(target - 1)  # logits are for item indices 1..num_items
    return np.array(x_rows, dtype=np.int32), np.array(y_rows, dtype=np.int32)


def build_predict_array(prompts: dict[int, np.ndarray], item_to_reduced: dict[int, int], seq_len: int) -> tuple[list[int], np.ndarray, list[list[int]]]:
    sids: list[int] = []
    x_rows: list[np.ndarray] = []
    seen_rows: list[list[int]] = []

    for sid, prompt in prompts.items():
        red = to_reduced_sequence(prompt, item_to_reduced)
        if not red:
            continue
        arr = np.zeros(seq_len, dtype=np.int32)
        arr[-len(red[-seq_len:]):] = np.array(red[-seq_len:], dtype=np.int32)
        sids.append(int(sid))
        x_rows.append(arr)
        seen_rows.append(red)

    if not x_rows:
        return [], np.empty((0, seq_len), dtype=np.int32), []
    return sids, np.array(x_rows, dtype=np.int32), seen_rows


def build_sasrec_like_model(num_items: int, seq_len: int, emb_dim: int, pretrained_item_embeddings: np.ndarray | None = None) -> keras.Model:
    inp = keras.Input(shape=(seq_len,), dtype="int32")

    item_emb_layer = keras.layers.Embedding(
        input_dim=num_items + 1,
        output_dim=emb_dim,
        mask_zero=True,
        name="item_emb",
    )

    x = item_emb_layer(inp)

    pos_idx = tf.range(start=0, limit=seq_len, delta=1)
    pos_emb_layer = keras.layers.Embedding(input_dim=seq_len, output_dim=emb_dim, name="pos_emb")
    pos_emb = pos_emb_layer(pos_idx)
    x = x + pos_emb

    key_dim = max(1, emb_dim // 2)
    attn = keras.layers.MultiHeadAttention(num_heads=2, key_dim=key_dim, dropout=0.2)
    attn_out = attn(x, x, use_causal_mask=True)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    ffn = keras.Sequential(
        [
            keras.layers.Dense(emb_dim * 2, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(emb_dim),
        ]
    )
    ffn_out = ffn(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_out)

    non_zero = tf.cast(tf.not_equal(inp, 0), tf.int32)
    last_idx = tf.maximum(tf.reduce_sum(non_zero, axis=1) - 1, 0)
    batch_idx = tf.range(tf.shape(inp)[0], dtype=tf.int32)
    gather_idx = tf.stack([batch_idx, last_idx], axis=1)
    last_hidden = tf.gather_nd(x, gather_idx)

    item_table = item_emb_layer.embeddings[1:]  # [num_items, emb_dim]
    logits = tf.matmul(last_hidden, item_table, transpose_b=True)

    model = keras.Model(inputs=inp, outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    )

    if pretrained_item_embeddings is not None:
        if pretrained_item_embeddings.shape != (num_items, emb_dim):
            raise ValueError(
                f"Expected pretrained embeddings shape {(num_items, emb_dim)}, got {pretrained_item_embeddings.shape}"
            )
        weights = item_emb_layer.get_weights()
        table = weights[0]
        table[1:, :] = pretrained_item_embeddings.astype(np.float32)
        item_emb_layer.set_weights([table])

    return model


def predict_topk(model: keras.Model, prompts: dict[int, np.ndarray], item_to_reduced: dict[int, int], reduced_to_item: dict[int, int], seq_len: int, top_k: int) -> dict[int, np.ndarray]:
    sids, x_arr, seen_rows = build_predict_array(prompts, item_to_reduced, seq_len)
    if len(sids) == 0:
        return {}

    logits = model.predict(x_arr, verbose=0, batch_size=1024)  # [B, num_items]
    preds: dict[int, np.ndarray] = {}

    for i, sid in enumerate(sids):
        scores = logits[i].astype(np.float64)
        seen = set(seen_rows[i])
        if seen:
            seen_idx = [r - 1 for r in seen if r > 0 and (r - 1) < len(scores)]
            scores[np.array(seen_idx, dtype=int)] = -1e12

        order = np.argsort(scores)[::-1][:top_k]
        rec_reduced = [int(o) + 1 for o in order]
        rec_orig = [reduced_to_item[r] for r in rec_reduced if r in reduced_to_item]
        preds[sid] = np.array(rec_orig, dtype=int)

    return preds


def predict_popular(prompts: dict[int, np.ndarray], train_counts: dict[int, int], top_k: int) -> dict[int, np.ndarray]:
    popular_items = [int(i) for i, _ in sorted(train_counts.items(), key=lambda kv: kv[1], reverse=True)]
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        seen = set(int(i) for i in prompt)
        recs = [i for i in popular_items if i not in seen][:top_k]
        out[int(sid)] = np.array(recs, dtype=int)
    return out


def session_embedding(prompt: np.ndarray, item_to_idx: dict[int, int], emb_norm: np.ndarray, decay: float = 1.0) -> np.ndarray | None:
    idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
    if not idxs:
        return None
    vecs = emb_norm[idxs]
    if len(vecs) == 1:
        return vecs[0]
    if decay == 1.0:
        out = vecs.mean(axis=0)
    else:
        n = len(vecs)
        weights = np.array([decay ** (n - i - 1) for i in range(n)], dtype=np.float32)
        weights = weights / np.sum(weights)
        out = np.average(vecs, axis=0, weights=weights)
    norm = np.linalg.norm(out)
    return out if norm == 0 else out / norm


def debias_weights(counts: np.ndarray, alpha: float, variant: str, ranks: np.ndarray, median_count: float) -> np.ndarray:
    c = np.maximum(counts.astype(np.float64), 1.0)
    if variant == "inverse":
        return 1.0 / np.power(c, alpha)
    if variant == "log_norm":
        return 1.0 / np.power(1.0 + np.log(c), alpha)
    if variant == "rank":
        return 1.0 / np.power(np.maximum(ranks.astype(np.float64), 1.0), alpha)
    if variant == "sigmoid":
        x = -alpha * ((c / max(median_count, 1.0)) - 1.0)
        return 1.0 / (1.0 + np.exp(-x))
    raise ValueError(f"Unknown variant: {variant}")


def rerank_popdebias(base_candidates: dict[int, np.ndarray], prompts: dict[int, np.ndarray], item_to_idx: dict[int, int], emb_norm: np.ndarray, item_counts: dict[int, int], alpha: float, top_k: int, variant: str = "inverse") -> dict[int, np.ndarray]:
    sorted_pop = sorted(item_counts.items(), key=lambda kv: kv[1], reverse=True)
    rank_map = {int(item): idx + 1 for idx, (item, _) in enumerate(sorted_pop)}
    median_count = float(np.median(np.array(list(item_counts.values()), dtype=np.float64)))

    out: dict[int, np.ndarray] = {}
    for sid, cand in base_candidates.items():
        q = session_embedding(prompts[sid], item_to_idx, emb_norm, decay=1.0)
        if q is None:
            out[sid] = np.array(cand[:top_k], dtype=int)
            continue

        cand = np.array([int(i) for i in cand if int(i) in item_to_idx], dtype=int)
        if cand.size == 0:
            out[sid] = np.array([], dtype=int)
            continue

        idxs = np.array([item_to_idx[int(i)] for i in cand], dtype=int)
        sims = emb_norm[idxs] @ q
        counts = np.array([item_counts.get(int(i), 1) for i in cand], dtype=np.float64)
        ranks = np.array([rank_map.get(int(i), len(rank_map) + 1) for i in cand], dtype=np.float64)
        weights = debias_weights(counts, alpha, variant, ranks, median_count)
        scores = sims * weights

        seen = set(int(i) for i in prompts[sid])
        if seen:
            mask = np.array([i not in seen for i in cand], dtype=bool)
            cand = cand[mask]
            scores = scores[mask]
        if cand.size == 0:
            out[sid] = np.array([], dtype=int)
            continue

        order = np.argsort(scores)[::-1]
        out[sid] = cand[order][:top_k].astype(int)
    return out


def predict_cold_branch(prompts: dict[int, np.ndarray], all_item_ids: np.ndarray, item_to_idx: dict[int, int], emb_norm: np.ndarray, top_k: int, decay: float) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        q = session_embedding(prompt, item_to_idx, emb_norm, decay=decay)
        if q is None:
            out[sid] = all_item_ids[:top_k].astype(int)
            continue
        scores = emb_norm @ q
        seen_idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
        if seen_idxs:
            scores[np.array(seen_idxs, dtype=int)] = -1e12
        order = np.argsort(scores)[::-1][:top_k]
        out[sid] = all_item_ids[order].astype(int)
    return out


def route_coldbridge(warm_preds: dict[int, np.ndarray], cold_preds: dict[int, np.ndarray], prompts: dict[int, np.ndarray], tau: int, top_k: int) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        out[sid] = (cold_preds[sid] if len(prompt) <= tau else warm_preds[sid])[:top_k]
    return out


def evaluate_at_k(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], k: int) -> tuple[float, float]:
    hrs = []
    ndcgs = []
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:k]]
        if target in rec:
            hrs.append(1.0)
            rank = rec.index(target) + 1
            ndcgs.append(1.0 / math.log2(rank + 1.0))
        else:
            hrs.append(0.0)
            ndcgs.append(0.0)
    return float(np.mean(ndcgs) if ndcgs else 0.0), float(np.mean(hrs) if hrs else 0.0)


def long_tail_hr10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], item_counts: dict[int, int], threshold: int) -> float:
    hits = 0
    total = 0
    for sid, gt in gts.items():
        target = int(gt[0])
        if item_counts.get(target, 0) >= threshold:
            continue
        total += 1
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        if target in rec:
            hits += 1
    return float(hits / total) if total > 0 else 0.0


def catalog_coverage10(preds: dict[int, np.ndarray], num_items: int) -> float:
    used = set()
    for rec in preds.values():
        used.update(int(i) for i in rec[:10])
    return float(len(used) / num_items) if num_items > 0 else 0.0


def serendipity10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], expected_popular_top10: list[int]) -> float:
    vals = []
    expected = set(expected_popular_top10)
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        vals.append(1.0 if (target in rec and target not in expected) else 0.0)
    return float(np.mean(vals) if vals else 0.0)


def ild10(preds: dict[int, np.ndarray], item_to_idx: dict[int, int], emb_norm: np.ndarray) -> float:
    vals = []
    for rec in preds.values():
        ids = [int(i) for i in rec[:10] if int(i) in item_to_idx]
        if len(ids) < 2:
            continue
        idxs = np.array([item_to_idx[i] for i in ids], dtype=int)
        vecs = emb_norm[idxs]
        sims = vecs @ vecs.T
        iu = np.triu_indices(len(ids), k=1)
        if len(iu[0]) == 0:
            continue
        vals.append(float(np.mean(1.0 - sims[iu])))
    return float(np.mean(vals) if vals else 0.0)


def segmented_hr10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], lengths: dict[int, int], cold_thr: int = 5) -> tuple[float, float]:
    cold_vals = []
    warm_vals = []
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        hit = 1.0 if target in rec else 0.0
        if lengths.get(sid, 0) <= cold_thr:
            cold_vals.append(hit)
        else:
            warm_vals.append(hit)
    cold_hr = float(np.mean(cold_vals) if cold_vals else 0.0)
    warm_hr = float(np.mean(warm_vals) if warm_vals else 0.0)
    return cold_hr, warm_hr


def evaluate_predictions(model_name: str, preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], item_counts: dict[int, int], num_items: int, emb_norm: np.ndarray, item_to_idx: dict[int, int], expected_pop_top10: list[int], alpha: float | None, tau: int | None, training_time: float, inference_time: float) -> dict[str, Any]:
    ndcg10, hr10 = evaluate_at_k(preds, gts, 10)
    ndcg20, hr20 = evaluate_at_k(preds, gts, 20)
    row = {
        "model_name": model_name,
        "alpha": alpha,
        "tau": tau,
        "NDCG@10": ndcg10,
        "NDCG@20": ndcg20,
        "HR@10": hr10,
        "HR@20": hr20,
        "LongTail_HR@10": long_tail_hr10(preds, gts, item_counts, LONG_TAIL_THRESHOLD),
        "CatalogCoverage": catalog_coverage10(preds, num_items),
        "Serendipity": serendipity10(preds, gts, expected_pop_top10),
        "ILD@10": ild10(preds, item_to_idx, emb_norm),
        "training_time_sec": float(training_time),
        "inference_time_sec": float(inference_time),
    }
    return row


def choose_best_joint(rows: list[dict[str, Any]]) -> dict[str, Any]:
    scored = []
    for row in rows:
        joint = 0.7 * float(row["NDCG@10"]) + 0.3 * float(row["LongTail_HR@10"])
        scored.append((joint, row))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def save_best_model(best_row: dict[str, Any], baseline_row: dict[str, Any] | None, output_path: Path) -> None:
    baseline_ndcg = float(baseline_row["NDCG@10"]) if baseline_row is not None else None
    best_ndcg = float(best_row["NDCG@10"])

    if baseline_ndcg and baseline_ndcg > 0:
        delta = ((best_ndcg - baseline_ndcg) / baseline_ndcg) * 100.0
        delta_text = f"{delta:+.2f}%"
        base_text = f"{baseline_ndcg:.6f}"
    else:
        delta_text = "N/A"
        base_text = "N/A"

    lines = [
        f"BEST MODEL: {best_row['model_name']}",
        f"Best Alpha: {best_row.get('alpha')}",
        f"Best Tau:   {best_row.get('tau')}",
        f"NDCG@10:    {best_ndcg:.6f} (vs baseline: {base_text}, delta: {delta_text})",
        f"HR@10:      {float(best_row['HR@10']):.6f}",
        f"LT-HR@10:   {float(best_row['LongTail_HR@10']):.6f}",
        f"ILD@10:     {float(best_row['ILD@10']):.6f}",
    ]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

In [ ]:
# Keras-3-safe override for the sequence model used below.
def build_sasrec_like_model(num_items: int, seq_len: int, emb_dim: int, pretrained_item_embeddings: np.ndarray | None = None) -> keras.Model:
    inp = keras.Input(shape=(seq_len,), dtype="int32")

    item_emb_layer = keras.layers.Embedding(
        input_dim=num_items + 1,
        output_dim=emb_dim,
        mask_zero=True,
        name="item_emb",
    )
    x = item_emb_layer(inp)

    attn = keras.layers.MultiHeadAttention(num_heads=2, key_dim=max(1, emb_dim // 2), dropout=0.2)
    attn_out = attn(x, x, use_causal_mask=True)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    ffn = keras.Sequential(
        [
            keras.layers.Dense(emb_dim * 2, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(emb_dim),
        ]
    )
    ffn_out = ffn(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_out)

    pooled = keras.layers.GlobalAveragePooling1D()(x)
    logits = keras.layers.Dense(num_items, name="logits")(pooled)

    model = keras.Model(inputs=inp, outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    )

    if pretrained_item_embeddings is not None:
        if pretrained_item_embeddings.shape != (num_items, emb_dim):
            raise ValueError(
                f"Expected pretrained embeddings shape {(num_items, emb_dim)}, got {pretrained_item_embeddings.shape}"
            )
        table = item_emb_layer.get_weights()[0]
        table[1:, :] = pretrained_item_embeddings.astype(np.float32)
        item_emb_layer.set_weights([table])

    return model

In [ ]:
# 1) Data preparation
udata_path, uitem_path = download_ml100k(DATA_DIR, DATASET_URL)
interactions = load_interactions(udata_path)
sessions_df = sessionize(interactions, SESSION_GAP_MINUTES, MIN_SESSION_LEN)
train_df, val_df, test_df = temporal_split(sessions_df, VAL_FRAC, TEST_FRAC)

val_prompts, val_gts, val_lengths = build_cases(val_df)
test_prompts, test_gts, test_lengths = build_cases(test_df)

print("Train sessions:", train_df["SessionId"].nunique())
print("Val sessions:", val_df["SessionId"].nunique())
print("Test sessions:", test_df["SessionId"].nunique())

# 2) Item metadata + BGE embeddings (CPU-safe)
item_metadata = load_item_metadata(uitem_path)
all_item_ids = np.array(sorted(sessions_df["ItemId"].unique().tolist()), dtype=int)
bge_path = EMB_DIR / "item_embeddings_bge_m3.npy"
bge_embeddings = encode_bge_cpu(
    item_ids=all_item_ids,
    item_text_df=item_metadata,
    output_npy=bge_path,
    batch_size=32,
    max_length=256,
 )
bge_embeddings = normalize_rows(bge_embeddings.astype(np.float32))

item_to_idx = {int(item): i for i, item in enumerate(all_item_ids.tolist())}
num_items = len(all_item_ids)

# 3) Reduced ID mapping for sequence model
item_to_reduced, reduced_to_item = build_id_maps(all_item_ids)

# 4) Build train examples
x_train, y_train = build_train_examples(train_df, item_to_reduced, SEQ_LEN)
print("Train examples:", x_train.shape, y_train.shape)

# 5) Prepare popularity statistics
item_counts = train_df["ItemId"].value_counts().to_dict()
popular_sorted = [int(i) for i, _ in sorted(item_counts.items(), key=lambda kv: kv[1], reverse=True)]
expected_pop_top10 = popular_sorted[:10]

results_rows: list[dict[str, Any]] = []

# 6) MostPopular baseline
t0 = time.perf_counter()
popular_preds_test = predict_popular(test_prompts, item_counts, TOP_K)
t1 = time.perf_counter()
row_pop = evaluate_predictions(
    model_name="MostPopular",
    preds=popular_preds_test,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=None,
    tau=None,
    training_time=0.0,
    inference_time=t1 - t0,
 )
results_rows.append(row_pop)

# 7) SASRec vanilla
sas_model = build_sasrec_like_model(num_items=num_items, seq_len=SEQ_LEN, emb_dim=EMB_DIM, pretrained_item_embeddings=None)
t0 = time.perf_counter()
sas_model.fit(x_train, y_train, epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
sas_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
sas_preds_test = predict_topk(sas_model, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, TOP_K)
sas_inf_time = time.perf_counter() - t0

row_sas = evaluate_predictions(
    model_name="SASRec",
    preds=sas_preds_test,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=None,
    tau=None,
    training_time=sas_train_time,
    inference_time=sas_inf_time,
 )
results_rows.append(row_sas)

# 8) BGE2SASRec (init with PCA-reduced BGE embeddings)
pca = PCA(n_components=EMB_DIM, random_state=SEED)
bge_reduced = pca.fit_transform(bge_embeddings).astype(np.float32)
bge_reduced = normalize_rows(bge_reduced)

bge2sas_model = build_sasrec_like_model(
    num_items=num_items,
    seq_len=SEQ_LEN,
    emb_dim=EMB_DIM,
    pretrained_item_embeddings=bge_reduced,
 )
t0 = time.perf_counter()
bge2sas_model.fit(x_train, y_train, epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
bge2sas_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
base_val_candidates = predict_topk(
    bge2sas_model, val_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
 )
base_test_candidates = predict_topk(
    bge2sas_model, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
 )
base_inf_time = time.perf_counter() - t0

bge2sas_test_top = {sid: rec[:TOP_K] for sid, rec in base_test_candidates.items()}
row_bge2sas = evaluate_predictions(
    model_name="BGE2SASRec",
    preds=bge2sas_test_top,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=None,
    tau=None,
    training_time=bge2sas_train_time,
    inference_time=base_inf_time,
 )
results_rows.append(row_bge2sas)

# 9) PopDebias alpha sweep
alpha_val_rows = []
alpha_test_rows = []
for alpha in ALPHAS:
    val_pred = rerank_popdebias(
        base_val_candidates, val_prompts, item_to_idx, bge_embeddings, item_counts, alpha, TOP_K, variant="inverse"
    )
    test_pred = rerank_popdebias(
        base_test_candidates, test_prompts, item_to_idx, bge_embeddings, item_counts, alpha, TOP_K, variant="inverse"
    )

    row_val = evaluate_predictions(
        model_name="PopDebias-BGE2SASRec-val",
        preds=val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=alpha,
        tau=None,
        training_time=0.0,
        inference_time=0.0,
    )
    alpha_val_rows.append(row_val)

    row_test = evaluate_predictions(
        model_name="PopDebias-BGE2SASRec",
        preds=test_pred,
        gts=test_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=alpha,
        tau=None,
        training_time=bge2sas_train_time,
        inference_time=base_inf_time,
    )
    alpha_test_rows.append(row_test)
    results_rows.append(row_test)

best_alpha_row = choose_best_joint(alpha_val_rows)
best_alpha = float(best_alpha_row["alpha"])

print("Best alpha:", best_alpha)

# 10) ColdBridge tau sweep
cold_val_uniform = predict_cold_branch(val_prompts, all_item_ids, item_to_idx, bge_embeddings, TOP_K, decay=1.0)
cold_test_uniform = predict_cold_branch(test_prompts, all_item_ids, item_to_idx, bge_embeddings, TOP_K, decay=1.0)
warm_val = {sid: rec[:TOP_K] for sid, rec in base_val_candidates.items()}
warm_test = {sid: rec[:TOP_K] for sid, rec in base_test_candidates.items()}

tau_val_rows = []
tau_test_rows = []
tau_stats = []
for tau in TAUS:
    val_pred = route_coldbridge(warm_val, cold_val_uniform, val_prompts, tau, TOP_K)
    test_pred = route_coldbridge(warm_test, cold_test_uniform, test_prompts, tau, TOP_K)

    row_val = evaluate_predictions(
        model_name="ColdBridge-BGE2SASRec-val",
        preds=val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=None,
        tau=tau,
        training_time=0.0,
        inference_time=0.0,
    )
    tau_val_rows.append(row_val)

    row_test = evaluate_predictions(
        model_name="ColdBridge-BGE2SASRec",
        preds=test_pred,
        gts=test_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=None,
        tau=tau,
        training_time=bge2sas_train_time,
        inference_time=base_inf_time,
    )
    tau_test_rows.append(row_test)
    results_rows.append(row_test)

    cold_hr, warm_hr = segmented_hr10(test_pred, test_gts, test_lengths, cold_thr=5)
    tau_stats.append({"tau": float(tau), "cold_hr10": cold_hr, "warm_hr10": warm_hr})

best_tau_row = choose_best_joint(tau_val_rows)
best_tau = int(best_tau_row["tau"])

print("Best tau:", best_tau)

# 11) Combined PopDebias + ColdBridge
warm_best_val = rerank_popdebias(base_val_candidates, val_prompts, item_to_idx, bge_embeddings, item_counts, best_alpha, TOP_K, variant="inverse")
warm_best_test = rerank_popdebias(base_test_candidates, test_prompts, item_to_idx, bge_embeddings, item_counts, best_alpha, TOP_K, variant="inverse")

combined_test = route_coldbridge(warm_best_test, cold_test_uniform, test_prompts, best_tau, TOP_K)
row_combined = evaluate_predictions(
    model_name="PopDebias-ColdBridge-BGE2SASRec",
    preds=combined_test,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=best_alpha,
    tau=best_tau,
    training_time=bge2sas_train_time,
    inference_time=base_inf_time,
 )
results_rows.append(row_combined)

# 12) Improvement 2: debias variants
variant_val_rows = []
for variant in ["log_norm", "rank", "sigmoid"]:
    for alpha in [0.1, 0.3, 0.5]:
        warm_val_variant = rerank_popdebias(base_val_candidates, val_prompts, item_to_idx, bge_embeddings, item_counts, alpha, TOP_K, variant=variant)
        warm_test_variant = rerank_popdebias(base_test_candidates, test_prompts, item_to_idx, bge_embeddings, item_counts, alpha, TOP_K, variant=variant)
        val_pred = route_coldbridge(warm_val_variant, cold_val_uniform, val_prompts, best_tau, TOP_K)
        test_pred = route_coldbridge(warm_test_variant, cold_test_uniform, test_prompts, best_tau, TOP_K)

        row_val = evaluate_predictions(
            model_name=f"Variant-{variant}-val",
            preds=val_pred,
            gts=val_gts,
            item_counts=item_counts,
            num_items=num_items,
            emb_norm=bge_embeddings,
            item_to_idx=item_to_idx,
            expected_pop_top10=expected_pop_top10,
            alpha=alpha,
            tau=best_tau,
            training_time=0.0,
            inference_time=0.0,
        )
        row_val["debias_variant"] = variant
        variant_val_rows.append(row_val)

        row_test = evaluate_predictions(
            model_name=f"PopDebias-ColdBridge-{variant}",
            preds=test_pred,
            gts=test_gts,
            item_counts=item_counts,
            num_items=num_items,
            emb_norm=bge_embeddings,
            item_to_idx=item_to_idx,
            expected_pop_top10=expected_pop_top10,
            alpha=alpha,
            tau=best_tau,
            training_time=bge2sas_train_time,
            inference_time=base_inf_time,
        )
        row_test["debias_variant"] = variant
        results_rows.append(row_test)

best_variant_row = choose_best_joint(variant_val_rows)
best_variant = str(best_variant_row["debias_variant"])
best_variant_alpha = float(best_variant_row["alpha"])

print("Best variant:", best_variant, "alpha:", best_variant_alpha)

# 13) Improvement 3: weighted cold branch
warm_test_variant_best = rerank_popdebias(
    base_test_candidates, test_prompts, item_to_idx, bge_embeddings, item_counts, best_variant_alpha, TOP_K, variant=best_variant
 )

decay_val_rows = []
for decay in DECAYS:
    cold_val = predict_cold_branch(val_prompts, all_item_ids, item_to_idx, bge_embeddings, TOP_K, decay=decay)
    cold_test = predict_cold_branch(test_prompts, all_item_ids, item_to_idx, bge_embeddings, TOP_K, decay=decay)

    warm_val_variant_best = rerank_popdebias(
        base_val_candidates, val_prompts, item_to_idx, bge_embeddings, item_counts, best_variant_alpha, TOP_K, variant=best_variant
    )

    val_pred = route_coldbridge(warm_val_variant_best, cold_val, val_prompts, best_tau, TOP_K)
    test_pred = route_coldbridge(warm_test_variant_best, cold_test, test_prompts, best_tau, TOP_K)

    row_val = evaluate_predictions(
        model_name="WeightedCold-val",
        preds=val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=best_variant_alpha,
        tau=best_tau,
        training_time=0.0,
        inference_time=0.0,
    )
    row_val["cold_decay"] = decay
    decay_val_rows.append(row_val)

    row_test = evaluate_predictions(
        model_name="PopDebias-ColdBridge-WeightedCold",
        preds=test_pred,
        gts=test_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=best_variant_alpha,
        tau=best_tau,
        training_time=bge2sas_train_time,
        inference_time=base_inf_time,
    )
    row_test["cold_decay"] = decay
    results_rows.append(row_test)

best_decay_row = choose_best_joint(decay_val_rows)
best_decay = float(best_decay_row["cold_decay"])

print("Best decay:", best_decay)

# 14) Full system
cold_test_best = predict_cold_branch(test_prompts, all_item_ids, item_to_idx, bge_embeddings, TOP_K, decay=best_decay)
full_preds = route_coldbridge(warm_test_variant_best, cold_test_best, test_prompts, best_tau, TOP_K)
row_full = evaluate_predictions(
    model_name="FULL_SYSTEM",
    preds=full_preds,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=best_variant_alpha,
    tau=best_tau,
    training_time=bge2sas_train_time,
    inference_time=base_inf_time,
 )
row_full["debias_variant"] = best_variant
row_full["cold_decay"] = best_decay
results_rows.append(row_full)

# 15) Save outputs
results_df = pd.DataFrame(results_rows)
base_cols = [
    "model_name", "alpha", "tau",
    "NDCG@10", "NDCG@20", "HR@10", "HR@20",
    "LongTail_HR@10", "CatalogCoverage", "Serendipity", "ILD@10",
    "training_time_sec", "inference_time_sec",
 ]
extra_cols = [c for c in ["debias_variant", "cold_decay"] if c in results_df.columns]
results_df = results_df[base_cols + extra_cols]
results_df = results_df.sort_values(["NDCG@10", "LongTail_HR@10"], ascending=False)

results_csv = RESULTS_DIR / "full_results.csv"
results_df.to_csv(results_csv, index=False)

best_row = choose_best_joint(results_df.to_dict("records"))
baseline_row = next((r for r in results_rows if r["model_name"] == "BGE2SASRec"), None)
best_txt = RESULTS_DIR / "best_model.txt"
save_best_model(best_row, baseline_row, best_txt)

# 16) Figures
alpha_df = pd.DataFrame(alpha_test_rows).sort_values("alpha")
plt.figure(figsize=(7, 5))
plt.plot(alpha_df["alpha"], alpha_df["NDCG@10"], marker="o", label="NDCG@10")
plt.plot(alpha_df["alpha"], alpha_df["LongTail_HR@10"], marker="s", label="LongTail_HR@10")
plt.xlabel("alpha")
plt.ylabel("metric")
plt.title("Alpha sweep - MovieLens-100K")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "alpha_sweep_movielens_100k.png", dpi=180)
plt.close()

tau_df = pd.DataFrame(tau_stats).sort_values("tau")
plt.figure(figsize=(7, 5))
plt.plot(tau_df["tau"], tau_df["cold_hr10"], marker="o", label="Cold HR@10")
plt.plot(tau_df["tau"], tau_df["warm_hr10"], marker="s", label="Warm HR@10")
plt.xlabel("tau")
plt.ylabel("HR@10")
plt.title("Tau sweep - MovieLens-100K")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "tau_sweep_movielens_100k.png", dpi=180)
plt.close()

base_cold, base_warm = segmented_hr10(bge2sas_test_top, test_gts, test_lengths, cold_thr=5)
full_cold, full_warm = segmented_hr10(full_preds, test_gts, test_lengths, cold_thr=5)
x = np.arange(2)
w = 0.35
plt.figure(figsize=(7, 5))
plt.bar(x - w / 2, [base_cold, base_warm], w, label="BGE2SASRec")
plt.bar(x + w / 2, [full_cold, full_warm], w, label="FULL_SYSTEM")
plt.xticks(x, ["Cold", "Warm"])

plt.ylabel("HR@10")
plt.title("Cold/Warm HR@10 gap")
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "cold_warm_gap_movielens_100k.png", dpi=180)
plt.close()

print("Done.")
print("Results CSV:", results_csv)
print("Best model:", best_txt)
print("Figures dir:", FIG_DIR)
display(results_df.head(20))

In [ ]:
# Optional: quick artifact preview after the main run cell
results_csv = RESULTS_DIR / "full_results.csv"
best_txt = RESULTS_DIR / "best_model.txt"
figures_dir = FIG_DIR

if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df.head(20))
else:
    print("Results file not found:", results_csv)

if best_txt.exists():
    print("=== best_model.txt ===")
    print(best_txt.read_text(encoding="utf-8"))
else:
    print("Best-model file not found:", best_txt)

if figures_dir.exists():
    print("Figures:")
    for p in sorted(figures_dir.glob("*.png")):
        print("-", p.name)

If Kaggle runtime is tight, lower `NUM_EPOCHS` (for example from `6` to `3`) and reduce `CANDIDATE_K` (for example from `300` to `150`) before re-running.